Task 1: Data Cleaning and Preprocessing

**Dataset:** Retail Store Sales (Dirty for Data Cleaning) - Kaggle
**Goal:** Take a messy, real-world sales dataset and turn it into a clean, structured dataset ready for analysis.

## Step 0: Import libraries and load the data

We need two libraries:
- **pandas** — to load and work with our data as a table (called a DataFrame)
- **numpy** — for numeric operations, mainly used when handling missing values

We then load our CSV file into a DataFrame called `df` (short for "dataframe").

In [7]:
import pandas as pd
import numpy as np


In [9]:
df = pd.read_csv(r'C:\Users\Admin\Downloads\archive\retail_store_sales.csv')


In [11]:
# Let's look at the first 5 rows to get a feel for the data
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [13]:
# How many rows and columns do we have?
print("Shape of dataset (rows, columns):", df.shape)


Shape of dataset (rows, columns): (12575, 11)


## Step 1: Identify and handle missing values

**Why this matters:** Missing values (also called "nulls" or "NaN") can break calculations, cause wrong conclusions, or crash machine learning models. Before fixing anything, we first need to *see* where the gaps are.

We use `.isnull().sum()` this checks every cell in every column and counts how many are empty.

In [16]:
# Check for missing values in each column
df.isnull().sum()

Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

**What we found:** Missing values exist in `Item`, `Price Per Unit`, `Quantity`, `Total Spent`, and `Discount Applied`.

**How we'll fix each one:**

1. **Item** — we don't have enough information to guess what the actual item was, so we'll label it `'Unknown'`. This way we keep the row instead of losing data, but we're honest that the value wasn't recorded.

2. **Discount Applied** — about a third of rows don't say whether a discount was used. Since we have no way to know for sure, we'll **assume `False`** (no discount) for missing entries. This is a reasonable default we document this assumption clearly so anyone reading our work understands the choice.

3. **Price Per Unit, Quantity, Total Spent** — these three numbers are mathematically connected:
   `Total Spent = Price Per Unit × Quantity`

   Instead of guessing these values randomly, we can use this formula to recover missing ones whenever we know the other two. This is much more accurate than filling with an average.

In [19]:
# Fix 1: Item — fill missing item codes with 'Unknown'
df['Item'] = df['Item'].fillna('Unknown')

In [21]:
# Fix 2: Discount Applied — fill missing values with False
df['Discount Applied'] = df['Discount Applied'].fillna(False)

C:\Users\Admin\AppData\Local\Temp\ipykernel_8884\2250915861.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Discount Applied'] = df['Discount Applied'].fillna(False)


In [23]:
# Fix 3: Recover Price / Quantity / Total Spent using their relationship
# Total Spent = Price Per Unit x Quantity

In [25]:
# (a) If Total Spent is missing but we know Price and Quantity, calculate it
mask = df['Total Spent'].isnull() 

In [37]:
# (a) If Total Spent is missing but we know Price and Quantity, calculate it
mask = df['Total Spent'].isnull() & df['Price Per Unit'].notnull() & df['Quantity'].notnull()
df.loc[mask, 'Total Spent'] = df.loc[mask, 'Price Per Unit'] * df.loc[mask, 'Quantity']


In [40]:
# (c) If Quantity is missing but we know Total Spent and Price Per Unit, calculate it
mask = df['Quantity'].isnull() & df['Total Spent'].notnull() & df['Price Per Unit'].notnull()
df.loc[mask, 'Quantity'] = df.loc[mask, 'Total Spent'] / df.loc[mask, 'Price Per Unit']


In [42]:
# Check how many nulls are left in these 3 columns
df[['Price Per Unit', 'Quantity', 'Total Spent']].isnull().sum()

Price Per Unit      0
Quantity          604
Total Spent       604
dtype: int64

**Why are there still a few nulls left?** 
Some rows are missing all three values at once Price, Quantity, AND Total Spent. When all three are blank, we can't use the formula (there's nothing to calculate from). For these rare leftover cases, we'll fill them with the **average (mean)** of that column. It's not perfect, but it's a safe, standard fallback for the small number of rows where no other information exists.

In [47]:
# Fill any remaining nulls (where all 3 were missing together) with the column mean
for col in ['Price Per Unit', 'Quantity', 'Total Spent']:
    df[col] = df[col].fillna(df[col].mean())

In [49]:
# Final check — this should show 0 for every column now
df.isnull().sum()

Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64

## Step 2: Remove duplicate rows

**Why this matters:** Duplicate rows can happen due to system errors, double data entry, or merging files incorrectly. They artificially inflate counts and totals, skewing any analysis we do later.

`.duplicated()` checks every row against every other row and flags exact copies. `.drop_duplicates()` removes them.

In [52]:
# How many exact duplicate rows do we have?
print("Duplicate rows found:", df.duplicated().sum())

Duplicate rows found: 0


In [54]:
# Remove them (if any)
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (12575, 11)


## Step 3: Standardize text values

**Why this matters:** The same value can be written in different ways `"male"`, `"Male"`, `"MALE"` and a computer treats these as three *different* categories unless we standardize them. This causes incorrect grouping and counting.

We'll check the text columns (`Category`, `Payment Method`, `Location`) and make their formatting consistent removing extra spaces and using Title Case (first letter capitalized).

In [58]:
# Look at the unique values before standardizing
print("Category:", df['Category'].unique())
print("Payment Method:", df['Payment Method'].unique())
print("Location:", df['Location'].unique())

Category: ['Patisserie' 'Milk Products' 'Butchers' 'Beverages' 'Food' 'Furniture'
 'Electric household essentials' 'Computers and electric accessories']
Payment Method: ['Digital Wallet' 'Credit Card' 'Cash']
Location: ['Online' 'In-store']


In [60]:
# Standardize: remove extra spaces, make Title Case
for col in ['Category', 'Payment Method', 'Location']:
    df[col] = df[col].str.strip().str.title()

# Confirm the cleanup
print("Category:", df['Category'].unique())
print("Payment Method:", df['Payment Method'].unique())
print("Location:", df['Location'].unique())

Category: ['Patisserie' 'Milk Products' 'Butchers' 'Beverages' 'Food' 'Furniture'
 'Electric Household Essentials' 'Computers And Electric Accessories']
Payment Method: ['Digital Wallet' 'Credit Card' 'Cash']
Location: ['Online' 'In-Store']


## Step 4: Convert date formats to a consistent type

**Why this matters:** Dates stored as plain text (strings) can't be sorted, filtered by range, or used in time-based calculations properly. We need to convert the column into an actual datetime type, then display it in one consistent format: **dd-mm-yyyy**.

In [63]:
# Check current format
print(df['Transaction Date'].head())
print(df['Transaction Date'].dtype)

0    2024-04-08
1    2023-07-23
2    2022-10-05
3    2022-05-07
4    2022-10-02
Name: Transaction Date, dtype: object
object


In [65]:
# Step 1: Convert text to a real datetime object
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])


In [67]:
# Step 2: Format it consistently as dd-mm-yyyy
df['Transaction Date'] = df['Transaction Date'].dt.strftime('%d-%m-%Y')

print(df['Transaction Date'].head())

0    08-04-2024
1    23-07-2023
2    05-10-2022
3    07-05-2022
4    02-10-2022
Name: Transaction Date, dtype: object


Transaction Date shows as object here that's fine since you formatted it as a dd-mm-yyyy string for display in Step 4, which converts it from datetime back to text. (Just know that if you need to filter or sort by date later, you'd reconvert with pd.to_datetime() first.)

## Step 5: Rename column headers to be clean and uniform

**Why this matters:** Column names like `Total Spent ` (with a trailing space) or `Order ID` (with a capital letter and space) are easy to mistype and inconsistent to work with in code. We'll convert every header to lowercase, with underscores instead of spaces, and no extra whitespace.

In [72]:
print("Before:", list(df.columns))

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print("After:", list(df.columns))

Before: ['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied']
After: ['transaction_id', 'customer_id', 'category', 'item', 'price_per_unit', 'quantity', 'total_spent', 'payment_method', 'location', 'transaction_date', 'discount_applied']


## Step 6: Check and fix data types

**Why this matters:** A column might *look* like numbers but actually be stored as text (called an "object" dtype in pandas). This breaks any math we try to do on it. We check each column's dtype and fix any mismatches e.g., quantity should be a whole number (int), price should be a decimal (float), and discount_applied should be a True/False value (boolean).

In [76]:
# Check current data types
df.dtypes

transaction_id       object
customer_id          object
category             object
item                 object
price_per_unit      float64
quantity            float64
total_spent         float64
payment_method       object
location             object
transaction_date     object
discount_applied       bool
dtype: object

In [80]:
# Fix data types
df['quantity'] = df['quantity'].astype(int)
df['price_per_unit'] = df['price_per_unit'].astype(float).round(2)
df['total_spent'] = df['total_spent'].astype(float).round(2)
df['discount_applied'] = df['discount_applied'].astype(bool)

# Confirm the fix
df.dtypes

transaction_id       object
customer_id          object
category             object
item                 object
price_per_unit      float64
quantity              int32
total_spent         float64
payment_method       object
location             object
transaction_date     object
discount_applied       bool
dtype: object

## Step 7 (Bonus): Check for outliers

**Why this matters:** Outliers are values that are unusually high or low compared to the rest of the data sometimes they're genuine (a big bulk order), and sometimes they're data entry errors. We use the **IQR method** (Interquartile Range) to flag them, then decide whether to keep or remove them.

**Decision:** The outliers found in `total_spent` are large but valid they come from genuine bulk purchases (price × quantity still matches correctly), not data entry mistakes. So we **keep** them rather than deleting real sales.

## Final Step: Save the cleaned dataset

Now that every issue has been fixed, we save our clean DataFrame to a new CSV file this is our final deliverable.

In [91]:
df.to_csv('cleaned_retail_store_sales.csv', index=False)
print("Cleaned dataset saved successfully!")
print("Final shape:", df.shape)
df.head()

Cleaned dataset saved successfully!
Final shape: (12575, 11)


,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10,185.0,Digital Wallet,Online,08-04-2024,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9,261.0,Digital Wallet,Online,23-07-2023,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2,43.0,Credit Card,Online,05-10-2022,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9,247.5,Credit Card,Online,07-05-2022,False
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7,87.5,Digital Wallet,Online,02-10-2022,False


## Conclusion

In this task, I took a raw, messy retail sales dataset (12,575 rows) and transformed it into a clean, analysis-ready dataset through the following steps:

1. **Identified and handled missing values** instead of blindly filling gaps, I used the mathematical relationship between `Price`, `Quantity`, and `Total Spent` to recover missing numbers accurately, and used sensible defaults (`'Unknown'`, `False`) for columns where no relationship existed.
2. **Removed duplicate rows** to avoid double-counting in future analysis.
3. **Standardized text formatting** across category, payment method, and location columns so similar values aren't treated as different groups.
4. **Converted dates into a consistent, usable format.**
5. **Cleaned up column headers** to be uniform and easy to reference in code.
6. **Fixed incorrect data types**, so numeric and boolean columns behave correctly in any future calculation.
7. **Checked for outliers** and made a judgment call to keep genuine high-value transactions rather than deleting valid data.

**What I learned:** Data cleaning isn't just about removing problems it's about making informed decisions at each step and being able to explain *why* you made them. Using relationships between columns (like Price × Quantity = Total) to recover missing data is far more reliable than guessing with averages, and should always be checked first before falling back to simpler methods.

The final cleaned dataset (`cleaned_retail_store_sales.csv`) is now ready for further analysis, visualization, or modelling.